# Engineered features

Create leakage-safe engineered features from the preprocessed artifacts produced by notebook 03. The output of this notebook is a new fold pickle plus matching engineered train/test CSVs for downstream single-model and stacking experiments.

Notebook 04 and earlier remain unchanged.

## 1. Notebook setup

### 1.1. Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

### 1.2. Run configuration

In [11]:
ENGINEERED_CV_FOLDS   = '../data/tmp/05-engineered-cv-folds.pkl'
ENGINEERED_TRAIN_DATA = '../data/tmp/05-engineered-train-data.csv'
ENGINEERED_TEST_DATA  = '../data/tmp/05-engineered-test-data.csv'

SOURCE_CV_FOLDS       = '../data/tmp/03-preprocessed-cv-folds.pkl'
SOURCE_TRAIN_DATA     = '../data/tmp/03-preprocessed-train-data.csv'
SOURCE_TEST_DATA      = '../data/tmp/03-preprocessed-test-data.csv'

SLEEP_BINS    = [0, 5, 7, 9, np.inf]
BMI_BINS      = [0, 18.5, 25.0, 30.0, np.inf]
STEP_BINS     = [0, 2500, 5000, 7500, np.inf]
EXERCISE_BINS = [0, 15, 30, 60, np.inf]
WATER_BINS    = [0, 1, 2, 3, np.inf]

WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99
RARE_FREQUENCY_THRESHOLD = 0.02

NUMERIC_FEATURE_CANDIDATES = [
    'sleep_duration',
    'bmi',
    'step_count',
    'exercise_duration',
    'water_intake',
    'heart_rate',
    'calorie_expenditure',
]

# Raw categorical columns from the original CSV are one-hot encoded in notebook 03.
# Reconstruct compact categorical views from these one-hot groups first, then apply
# frequency, rarity, cross, and group-stat engineering on the derived columns.
DERIVED_CATEGORICAL_GROUPS = {
    'stress_level': ['stress_level_0', 'stress_level_1', 'stress_level_2'],
    'sleep_quality': ['sleep_quality_0', 'sleep_quality_1', 'sleep_quality_2'],
    'diet_type': ['diet_type_0', 'diet_type_1', 'diet_type_2'],
    'physical_activity_level': [
        'physical_activity_level_0',
        'physical_activity_level_1',
        'physical_activity_level_2',
    ],
    'smoking_alcohol': ['smoking_alcohol_0', 'smoking_alcohol_1', 'smoking_alcohol_2'],
    'gender': ['gender_0', 'gender_1', 'gender_2'],
}

FREQUENCY_FEATURE_CANDIDATES = [
    'stress_level',
    'sleep_quality',
    'diet_type',
    'physical_activity_level',
    'smoking_alcohol',
    'gender',
]

# Categorical crosses used for frequency and rarity signals.
CATEGORICAL_CROSS_PAIRS = [
    ('stress_level', 'sleep_quality'),
    ('diet_type', 'smoking_alcohol'),
    ('physical_activity_level', 'sleep_quality'),
    ('gender', 'stress_level'),
]

# Group-stat and residual features (fit on training split only).
GROUP_STAT_CATEGORICAL_COLUMNS = [
    'stress_level',
    'sleep_quality',
    'diet_type',
    'physical_activity_level',
    'smoking_alcohol',
    'gender',
]
GROUP_STAT_NUMERIC_COLUMNS = [
    'sleep_duration',
    'heart_rate',
    'step_count',
    'calorie_expenditure',
]
GROUP_STAT_METRICS = ['mean', 'std', 'median']

## 2. Load artifacts from notebook 03

In [12]:
with open(SOURCE_CV_FOLDS, 'rb') as handle:
    cv_folds = pickle.load(handle)

train_df = pd.read_csv(SOURCE_TRAIN_DATA)
test_df = pd.read_csv(SOURCE_TEST_DATA)

base_feature_columns = [column for column in train_df.columns if column != 'health_condition']
missing_indicator_columns = [column for column in base_feature_columns if column.endswith('_missing')]

print(f'Loaded {len(cv_folds)} folds')
print(f'Preprocessed train shape: {train_df.shape}')
print(f'Preprocessed test shape:  {test_df.shape}')
print(f'Missing indicator columns: {missing_indicator_columns}')

Loaded 10 folds
Preprocessed train shape: (690088, 33)
Preprocessed test shape:  (295753, 33)
Missing indicator columns: ['sleep_duration_missing', 'heart_rate_missing', 'bmi_missing', 'calorie_expenditure_missing', 'step_count_missing', 'exercise_duration_missing', 'water_intake_missing']


## 3. Build engineered features

Feature creation is split into sequential sections so each transformation family is easier to read, maintain, and profile.

In [13]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def cross_feature_name(left_column, right_column):
    return f'{left_column}_x_{right_column}'


def add_derived_categorical_features(df, categorical_groups):
    engineered = df.copy()

    for derived_column, source_columns in categorical_groups.items():
        present_columns = [column for column in source_columns if column in engineered.columns]

        if len(present_columns) < 2:
            continue

        value_levels = []

        for column in present_columns:
            try:
                value_levels.append(int(column.rsplit('_', 1)[1]))
            except (IndexError, ValueError):
                value_levels.append(len(value_levels))

        row_values = engineered[present_columns].to_numpy()
        argmax_indices = row_values.argmax(axis=1)
        max_values = row_values.max(axis=1)

        derived_values = np.array(value_levels, dtype=int)[argmax_indices]
        derived_values[max_values <= 0] = -1

        engineered[derived_column] = derived_values

    return engineered


def fit_winsor_limits(reference_df, columns, lower_q=0.01, upper_q=0.99):
    limits = {}
    for column in columns:
        if column not in reference_df.columns:
            continue
        lower = float(reference_df[column].quantile(lower_q))
        upper = float(reference_df[column].quantile(upper_q))
        limits[column] = (lower, upper)
    return limits


def add_categorical_cross_features(df, cross_pairs):
    engineered = df.copy()
    added_cross_columns = []

    for left_column, right_column in cross_pairs:
        if left_column not in engineered.columns or right_column not in engineered.columns:
            continue

        cross_column = cross_feature_name(left_column, right_column)
        left_series = engineered[left_column].astype(object).where(engineered[left_column].notna(), 'MISSING')
        right_series = engineered[right_column].astype(object).where(engineered[right_column].notna(), 'MISSING')
        engineered[cross_column] = left_series.astype(str) + '__x__' + right_series.astype(str)
        added_cross_columns.append(cross_column)

    return engineered, added_cross_columns


def fit_frequency_maps(reference_df, columns):
    maps = {}
    for column in columns:
        if column not in reference_df.columns:
            continue
        maps[column] = reference_df[column].value_counts(normalize=True, dropna=False)
    return maps


def fit_group_stat_maps(reference_df, categorical_columns, numeric_columns, metrics):
    group_maps = {}
    global_stats = {}

    available_categorical = [column for column in categorical_columns if column in reference_df.columns]
    available_numeric = [column for column in numeric_columns if column in reference_df.columns]

    for numeric_column in available_numeric:
        global_stats[numeric_column] = {
            'mean': float(reference_df[numeric_column].mean()),
            'std': float(reference_df[numeric_column].std(ddof=0)),
            'median': float(reference_df[numeric_column].median()),
        }
        if pd.isna(global_stats[numeric_column]['std']):
            global_stats[numeric_column]['std'] = 0.0

    for categorical_column in available_categorical:
        grouped_stats = reference_df.groupby(categorical_column, dropna=False)[available_numeric].agg(metrics)

        for numeric_column in available_numeric:
            for metric in metrics:
                mapping_key = (categorical_column, numeric_column, metric)
                mapping_series = grouped_stats[(numeric_column, metric)]
                group_maps[mapping_key] = mapping_series.to_dict()

    return group_maps, global_stats


def summarize_added_columns(added_columns):
    summary = {
        'cross_features': 0,
        'frequency_features': 0,
        'rarity_flags': 0,
        'group_stats': 0,
        'group_residuals': 0,
        'log_transforms': 0,
        'sqrt_transforms': 0,
        'winsorized': 0,
        'bands': 0,
        'other': 0,
    }

    for column in added_columns:
        if '__x__' in column or '_x_' in column:
            summary['cross_features'] += 1
        elif column.endswith('_freq'):
            summary['frequency_features'] += 1
        elif column.endswith('_is_rare'):
            summary['rarity_flags'] += 1
        elif '_by_' in column and column.endswith(('_mean', '_std', '_median')):
            summary['group_stats'] += 1
        elif '_minus_mean_by_' in column or '_minus_median_by_' in column:
            summary['group_residuals'] += 1
        elif column.startswith('log1p_'):
            summary['log_transforms'] += 1
        elif column.startswith('sqrt_'):
            summary['sqrt_transforms'] += 1
        elif column.endswith('_winsor'):
            summary['winsorized'] += 1
        elif column.endswith('_band'):
            summary['bands'] += 1
        else:
            summary['other'] += 1

    return summary


print('Utility helpers loaded for sections 3.1-3.4')

Utility helpers loaded for sections 3.1-3.4


### 3.1. Base ratio, interaction, and transformation features
Define deterministic row-level transforms and interactions applied before frequency or grouped context features.

In [16]:
def add_base_engineered_features(df, missing_columns, numeric_columns, winsor_limits):
    engineered = df.copy()
    new_columns = {}

    if {'step_count', 'exercise_duration'}.issubset(engineered.columns):
        new_columns['step_rate'] = safe_divide(engineered['step_count'], engineered['exercise_duration'])
        new_columns['activity_intensity'] = safe_divide(engineered['step_count'], engineered['exercise_duration'] + 1.0)
        new_columns['exercise_x_steps'] = engineered['exercise_duration'] * engineered['step_count']

    if {'calorie_expenditure', 'exercise_duration'}.issubset(engineered.columns):
        new_columns['calorie_rate'] = safe_divide(engineered['calorie_expenditure'], engineered['exercise_duration'])

    if {'water_intake', 'exercise_duration'}.issubset(engineered.columns):
        new_columns['water_rate'] = safe_divide(engineered['water_intake'], engineered['exercise_duration'])

    if {'sleep_duration', 'exercise_duration'}.issubset(engineered.columns):
        new_columns['sleep_to_activity'] = safe_divide(engineered['sleep_duration'], engineered['exercise_duration'])

    if {'calorie_expenditure', 'bmi'}.issubset(engineered.columns):
        new_columns['calorie_density'] = safe_divide(engineered['calorie_expenditure'], engineered['bmi'] + 1.0)

    if {'sleep_duration', 'heart_rate'}.issubset(engineered.columns):
        new_columns['recovery_ratio'] = safe_divide(engineered['sleep_duration'], engineered['heart_rate'] + 1.0)

    if {'water_intake', 'sleep_duration'}.issubset(engineered.columns):
        new_columns['hydration_sleep_ratio'] = safe_divide(engineered['water_intake'], engineered['sleep_duration'] + 1.0)

    if {'heart_rate', 'exercise_duration'}.issubset(engineered.columns):
        new_columns['heart_rate_x_exercise'] = engineered['heart_rate'] * engineered['exercise_duration']

    if {'bmi', 'heart_rate'}.issubset(engineered.columns):
        new_columns['bmi_x_heart_rate'] = engineered['bmi'] * engineered['heart_rate']

    if {'step_count', 'calorie_expenditure'}.issubset(engineered.columns):
        new_columns['step_to_calorie_ratio'] = safe_divide(engineered['step_count'], engineered['calorie_expenditure'] + 1.0)

    if missing_columns:
        present_missing_columns = [column for column in missing_columns if column in engineered.columns]
        new_columns['missing_indicator_count'] = engineered[present_missing_columns].sum(axis=1) if present_missing_columns else 0.0
    else:
        new_columns['missing_indicator_count'] = 0.0

    if 'sleep_duration' in engineered.columns:
        new_columns['sleep_duration_band'] = pd.cut(
            engineered['sleep_duration'],
            bins=SLEEP_BINS,
            labels=False,
            include_lowest=True,
        ).fillna(-1).astype(int)

    if 'bmi' in engineered.columns:
        new_columns['bmi_band'] = pd.cut(
            engineered['bmi'],
            bins=BMI_BINS,
            labels=False,
            include_lowest=True,
        ).fillna(-1).astype(int)

    if 'step_count' in engineered.columns:
        new_columns['step_count_band'] = pd.cut(
            engineered['step_count'],
            bins=STEP_BINS,
            labels=False,
            include_lowest=True,
        ).fillna(-1).astype(int)

    if 'exercise_duration' in engineered.columns:
        new_columns['exercise_duration_band'] = pd.cut(
            engineered['exercise_duration'],
            bins=EXERCISE_BINS,
            labels=False,
            include_lowest=True,
        ).fillna(-1).astype(int)

    if 'water_intake' in engineered.columns:
        new_columns['water_intake_band'] = pd.cut(
            engineered['water_intake'],
            bins=WATER_BINS,
            labels=False,
            include_lowest=True,
        ).fillna(-1).astype(int)

    for column in numeric_columns:
        if column not in engineered.columns:
            continue
        clipped = engineered[column].clip(lower=0)
        new_columns[f'log1p_{column}'] = np.log1p(clipped)
        new_columns[f'sqrt_{column}'] = np.sqrt(clipped)

    for column, (lower, upper) in winsor_limits.items():
        if column in engineered.columns:
            new_columns[f'{column}_winsor'] = engineered[column].clip(lower=lower, upper=upper)

    if new_columns:
        engineered = pd.concat([engineered, pd.DataFrame(new_columns, index=engineered.index)], axis=1)

    return engineered


print('Section 3.1 ready: base ratio/interaction/transformation features')

Section 3.1 ready: base ratio/interaction/transformation features


### 3.2. Frequency and rarity features
Map category prevalence from the training split, then flag uncommon values with compact binary indicators.

In [17]:
def apply_frequency_features(df, frequency_maps, rare_frequency_threshold=0.02):
    engineered = df.copy()
    new_columns = {}

    for column, value_frequencies in frequency_maps.items():
        if column not in engineered.columns:
            continue

        frequency_column = f'{column}_freq'
        rarity_column = f'{column}_is_rare'
        mapped_frequency = engineered[column].map(value_frequencies).fillna(0.0).astype(float)

        new_columns[frequency_column] = mapped_frequency
        new_columns[rarity_column] = (mapped_frequency < rare_frequency_threshold).astype(int)

    if new_columns:
        engineered = pd.concat([engineered, pd.DataFrame(new_columns, index=engineered.index)], axis=1)

    return engineered


print('Section 3.2 ready: frequency and rarity features')

Section 3.2 ready: frequency and rarity features


### 3.3. Group statistics and residual features
Add group-level context statistics and residual-from-group signals using training-split aggregates only.

In [18]:
def apply_group_stat_features(df, group_maps, global_stats):
    engineered = df.copy()
    new_columns = {}

    for (categorical_column, numeric_column, metric), value_map in group_maps.items():
        if categorical_column not in engineered.columns or numeric_column not in engineered.columns:
            continue

        feature_name = f'{numeric_column}_by_{categorical_column}_{metric}'
        default_value = global_stats[numeric_column][metric]
        mapped = pd.to_numeric(engineered[categorical_column].map(value_map), errors='coerce').fillna(default_value)
        new_columns[feature_name] = mapped

        if metric in {'mean', 'median'}:
            residual_name = f'{numeric_column}_minus_{metric}_by_{categorical_column}'
            new_columns[residual_name] = engineered[numeric_column] - mapped

    if new_columns:
        engineered = pd.concat([engineered, pd.DataFrame(new_columns, index=engineered.index)], axis=1)

    return engineered


print('Section 3.3 ready: group statistics and residual features')

Section 3.3 ready: group statistics and residual features


### 3.4. Fold-wise feature engineering run
Apply all feature groups in sequence for each fold and for full train/test artifacts with concise progress output.

In [14]:
def transform_with_fitted_features(reference_df, target_df, missing_columns, numeric_columns, frequency_columns):

    reference_with_categories = add_derived_categorical_features(reference_df, DERIVED_CATEGORICAL_GROUPS)
    target_with_categories = add_derived_categorical_features(target_df, DERIVED_CATEGORICAL_GROUPS)

    reference_with_crosses, cross_columns = add_categorical_cross_features(reference_with_categories, CATEGORICAL_CROSS_PAIRS)
    target_with_crosses, _ = add_categorical_cross_features(target_with_categories, CATEGORICAL_CROSS_PAIRS)

    available_numeric_columns = [column for column in numeric_columns if column in reference_with_crosses.columns]
    available_frequency_columns = [column for column in frequency_columns if column in reference_with_crosses.columns]
    frequency_columns_with_crosses = available_frequency_columns + cross_columns

    winsor_limits = fit_winsor_limits(
        reference_with_crosses,
        columns=available_numeric_columns,
        lower_q=WINSOR_LOWER_Q,
        upper_q=WINSOR_UPPER_Q,
    )

    frequency_maps = fit_frequency_maps(reference_with_crosses, frequency_columns_with_crosses)

    group_maps, global_stats = fit_group_stat_maps(
        reference_with_crosses,
        categorical_columns=GROUP_STAT_CATEGORICAL_COLUMNS,
        numeric_columns=GROUP_STAT_NUMERIC_COLUMNS,
        metrics=GROUP_STAT_METRICS,
    )

    transformed = add_base_engineered_features(
        target_with_crosses,
        missing_columns=missing_columns,
        numeric_columns=available_numeric_columns,
        winsor_limits=winsor_limits,
    )

    transformed = apply_frequency_features(
        transformed,
        frequency_maps=frequency_maps,
        rare_frequency_threshold=RARE_FREQUENCY_THRESHOLD,
    )

    transformed = apply_group_stat_features(
        transformed,
        group_maps=group_maps,
        global_stats=global_stats,
    )

    return transformed

In [19]:
engineered_folds = []
total_folds = len(cv_folds)
print(f'3.4 fold-wise engineering started ({total_folds} folds)')

for fold_index, fold in enumerate(cv_folds, start=1):
    engineered_fold = fold.copy()
    engineered_fold['x_train'] = transform_with_fitted_features(
        reference_df=fold['x_train'],
        target_df=fold['x_train'],
        missing_columns=missing_indicator_columns,
        numeric_columns=NUMERIC_FEATURE_CANDIDATES,
        frequency_columns=FREQUENCY_FEATURE_CANDIDATES,
    )
    engineered_fold['x_validation'] = transform_with_fitted_features(
        reference_df=fold['x_train'],
        target_df=fold['x_validation'],
        missing_columns=missing_indicator_columns,
        numeric_columns=NUMERIC_FEATURE_CANDIDATES,
        frequency_columns=FREQUENCY_FEATURE_CANDIDATES,
    )
    engineered_folds.append(engineered_fold)

    if fold_index == 1 or fold_index % 2 == 0 or fold_index == total_folds:
        print(f'  progress {fold_index}/{total_folds}')

print('3.4 building full train/test engineered datasets')

engineered_train_df = transform_with_fitted_features(
    reference_df=train_df,
    target_df=train_df,
    missing_columns=missing_indicator_columns,
    numeric_columns=NUMERIC_FEATURE_CANDIDATES,
    frequency_columns=FREQUENCY_FEATURE_CANDIDATES,
)

engineered_test_df = transform_with_fitted_features(
    reference_df=train_df,
    target_df=test_df,
    missing_columns=missing_indicator_columns,
    numeric_columns=NUMERIC_FEATURE_CANDIDATES,
    frequency_columns=FREQUENCY_FEATURE_CANDIDATES,
)

added_columns = [column for column in engineered_train_df.columns if column not in train_df.columns]
feature_summary = summarize_added_columns(added_columns)

print(f'Engineered train shape: {engineered_train_df.shape}')
print(f'Engineered test shape:  {engineered_test_df.shape}')
print(f'Added engineered columns: {len(added_columns)}')
print('Feature category summary:')

for key, value in feature_summary.items():
    print(f'  {key}: {value}')

print(f'Sample added columns (first 15): {added_columns[:15]}')

3.4 fold-wise engineering started (10 folds)
  progress 1/10
  progress 2/10
  progress 4/10
  progress 6/10
  progress 8/10
  progress 10/10
3.4 building full train/test engineered datasets
Engineered train shape: (690088, 222)
Engineered test shape:  (295753, 222)
Added engineered columns: 189
Feature category summary:
  cross_features: 15
  frequency_features: 6
  rarity_flags: 6
  group_stats: 72
  group_residuals: 48
  log_transforms: 7
  sqrt_transforms: 7
  winsorized: 7
  bands: 5
  other: 16
Sample added columns (first 15): ['stress_level', 'sleep_quality', 'diet_type', 'physical_activity_level', 'smoking_alcohol', 'gender', 'stress_level_x_sleep_quality', 'diet_type_x_smoking_alcohol', 'physical_activity_level_x_sleep_quality', 'gender_x_stress_level', 'step_rate', 'activity_intensity', 'exercise_x_steps', 'calorie_rate', 'water_rate']


## 4. Save engineered artifacts

In [10]:
with open(ENGINEERED_CV_FOLDS, 'wb') as handle:
    pickle.dump(engineered_folds, handle)

engineered_train_df.to_csv(ENGINEERED_TRAIN_DATA, index=False)
engineered_test_df.to_csv(ENGINEERED_TEST_DATA, index=False)

print('Saved engineered artifacts:')
print(f'  {ENGINEERED_CV_FOLDS}')
print(f'  {ENGINEERED_TRAIN_DATA}')
print(f'  {ENGINEERED_TEST_DATA}')

Saved engineered artifacts:
  ../data/tmp/05-engineered-cv-folds.pkl
  ../data/tmp/05-engineered-train-data.csv
  ../data/tmp/05-engineered-test-data.csv
